Phase 5: **Business SQL Problems**.

No more "use CTE" or "use window function." From now on, you decide the approach, just like in an interview.

We'll also start using the other tables in our dataset.

**Interview Hint:-
Before you start writing SQL, spend 30 seconds asking yourself**:

* Which table should drive the query?
* Do I need row-level data or aggregated data?
* Do I need a window function, or is GROUP BY enough?
* Do I need a CTE at all?

Sometimes the best SQL is the simplest SQL. That's an important lesson for interviews.

=======================================================================================================

Q053 ⭐⭐⭐⭐ (Customers & Orders)
Business Scenario

The Sales Head wants to identify valuable customers.

A valuable customer is one who has placed more than one order.

The report should show:

* Customer ID
* Customer Name
* Total Orders
* First Order Date
* Latest Order Date

Sort by:
* Total Orders DESC,
* Latest Order Date DESC

Business Rules:-
*   Include only customers having more than one order.
*   Ignore customers who have never placed an order.
*   If a customer placed 10 orders, count all 10.

In [0]:
%sql
select o.customer_id, c.customer_name, count(o.order_id) as total_orders, min(o.order_date) as first_order_date, max(o.order_date) as last_order_date
    from sql_interview.orders o
        inner join sql_interview.customers c
            on o.customer_id = c.customer_id
        where o.order_status = 'Delivered'
                and c.is_active = 'Y'
    group by o.customer_id, c.customer_name
        having count(order_id) > 1
    order by total_orders desc, 
                last_order_date desc
        

In [0]:
%sql

-- EXPLAIN FORMATTED
with active_customers as (
    select customer_id, customer_name
        from sql_interview.customers
            where is_active = 'Y'
),
customer_orders as (
    select o.order_id, ac.customer_id, ac.customer_name, o.order_date,
            count(o.order_id) over(partition by o.customer_id) as total_orders,
            first_value(order_date) over(partition by o.customer_id order by order_date) as first_order_date,
            first_value(order_date) over(partition by o.customer_id order by order_date desc) as last_order_date
        from sql_interview.orders o
            inner join active_customers ac
                on o.customer_id = ac.customer_id
    where o.order_status = 'Delivered'
),
customer_report as (
    select order_id, customer_id, customer_name, order_date, total_orders, first_order_date, last_order_date,
            row_number() over(partition by customer_id order by total_orders desc,
                                        order_date desc) as row_num
        from customer_orders
    where total_orders > 1
)
select customer_id, customer_name, total_orders, first_order_date, last_order_date
    from customer_report
where row_num = 1

Q054 ⭐⭐⭐⭐ (Orders + Order Items + Products)
Business Scenario

The Sales Director wants to identify the Top 5 best-selling products based on revenue.

Revenue is calculated as:
* quantity × unit_price

Business Rules:
* Consider only Delivered orders.
* Revenue = quantity * unit_price.
* If the same product appears in multiple orders, include all sales.
* Return only the Top 5 products.

In [0]:
%sql

with products_revenue as (
    select oi.product_id,sum( oi.quantity) as total_quantity_sold, sum(oi.quantity * oi.unit_price) as total_revenue
        from sql_interview.order_items oi
            inner join sql_interview.orders o
                on oi.order_id = o.order_id
        where o.order_status = 'Delivered'

            group by oi.product_id
),
revenue_ranking as (
    select pr.product_id, p.product_name, pr.total_quantity_sold, pr.total_revenue,
        dense_rank() over(order by pr.total_revenue desc) as revenue_rank
    from products_revenue pr
        inner join sql_interview.products p
            on pr.product_id = p.product_id
)
select * from revenue_ranking
    where revenue_rank <= 5




Q056 ⭐⭐⭐⭐⭐ (Products + Categories + Order Items + Orders)

Business Scenario:

| The Product Management team wants to know:
* Which product category generates the highest revenue?

Business Rules:
* Consider only Delivered orders.
* Revenue = quantity × unit_price
* Aggregate revenue at the category level.

Return:
* Category ID
* Category Name
* Number of Products Sold
* Total Revenue

Sort by:
* Total Revenue DESC

In [0]:
%sql
select *
    from sql_interview.categories


In [0]:
%sql
select *
    from sql_interview.products

In [0]:
%sql
select *
    from sql_interview.orders

In [0]:
%sql

-- Follow up 1
-- "Show the percentage contribution of each category to total revenue."

with category_sales as (
    select  c.category_id, c.category_name, sum(oi.quantity) as products_sold_count, sum(oi.quantity * oi.unit_price) as total_revenue
        from sql_interview.order_items oi
            inner join sql_interview.orders o
                on oi.order_id = o.order_id
            
            inner join sql_interview.products p
                on oi.product_id = p.product_id
            
            inner join sql_interview.categories c
                on p.category_id = c.category_id

        where o.order_status = 'Delivered'

        group by c.category_id, c.category_name
),
category_ranking as (
    select *,
        (total_revenue * 100) / sum(total_revenue) over() as pct_revenue,
        dense_rank() over(order by total_revenue desc) as revenue_rank
    from category_sales
)
select category_id, category_name, products_sold_count, total_revenue, 
        concat (cast(pct_revenue as decimal(10,2)), '%', ' ') as pct_contribution
    from category_ranking
        where revenue_rank <= 3

    

In [0]:
%sql
-- Follow up 2
-- "Show the highest revenue category for each year."

with category_sales as (
    select year(o.order_date) as order_year, c.category_id, c.category_name, sum(oi.quantity) as products_sold_count, sum(oi.quantity * oi.unit_price) as total_revenue
        from sql_interview.order_items oi
            inner join sql_interview.orders o
                on oi.order_id = o.order_id
            
            inner join sql_interview.products p
                on oi.product_id = p.product_id
            
            inner join sql_interview.categories c
                on p.category_id = c.category_id

        where o.order_status = 'Delivered'

        group by c.category_id, c.category_name, order_year
),
category_ranking as (
    select *,
            dense_rank() over(partition by order_year order by total_revenue desc) as revenue_rank
        from category_sales
)
select order_year, category_id, category_name, products_sold_count, total_revenue
    from category_ranking
        where revenue_rank = 1


Q057 ⭐⭐⭐⭐⭐⭐ (Customer's Favorite Category)
Business Scenario

The Marketing team wants to personalize recommendations.

They define a customer's favorite category as:

* The category in which the customer has spent the highest amount.

**The business requirement was**:

**Return**:
* Customer ID
* Customer Name
* Category ID
* Category Name
* Total Spending

Business Rules:
* Consider only Delivered orders.
* Spending = SUM(quantity × unit_price).
* One customer may purchase from multiple categories.
* Return one row per customer.
* If multiple categories tie for the highest spending, return all of them.

**Things to Think About**

Before writing SQL, ask yourself:

* What is the lowest level of data?
* At what grain should you aggregate first?
* When should you join customers?
* When should you join categories?
* Where should the ranking happen?
* Which ranking function is appropriate?

In [0]:
%sql

with customer_spendings as (
    select o.customer_id, c.customer_name, sum(oi.quantity * oi.unit_price) as highest_spent, p.category_id, cat.category_name
        from sql_interview.orders o
            inner join sql_interview.order_items oi
                on o.order_id = oi.order_id

            inner join sql_interview.products p
                on oi.product_id = p.product_id

            inner join sql_interview.categories cat
                on p.category_id = cat.category_id

            inner join sql_interview.customers c
                on o.customer_id = c.customer_id

            where o.order_status = 'Delivered'

            group by o.customer_id,
                    c.customer_name,
                    p.category_id,
                    cat.category_name

),
category_ranking as (
    select customer_id, customer_name, category_id, category_name, highest_spent,
            dense_rank() over(partition by customer_id order by highest_spent desc) as top_rank
        from customer_spendings
)
select customer_id, customer_name, category_id, category_name, highest_spent
    from category_ranking
        where top_rank = 1


Q058 ⭐⭐⭐⭐⭐⭐ (Hard - Consecutive Purchase Streak)

This is a classic interview question because it combines business logic with window functions.

**Business Scenario**

The retention team wants to identify **customers who made purchases on three or more consecutive days**.

**Business Rules**:
* Consider only **Delivered** orders.
* If a customer places multiple orders on the same day, count that as **one purchase day**.
* Find customers with **streaks of at least 3 consecutive calendar days**.
* Return **every qualifying streak** (a customer can have more than one).

**Tables**
* orders
* customers


**Things to Think About**

Don't jump into coding immediately. Ask yourself:

* How do I eliminate multiple orders on the same day?
* How can I identify consecutive dates?
* What should I partition by?
* How can I assign the same identifier to rows belonging to the same streak?
* At what stage do I calculate the streak length?

This is a well-known advanced SQL interview problem because it tests whether you can recognize and solve **a gaps and islands** pattern rather than just apply syntax mechanically.

In [0]:
%sql

with customer_purchase as (
    select *, 
        row_number() over(partition by customer_id, order_date order by order_date) as purchase_per_day
    from sql_interview.orders
        where order_status = 'Delivered'
),
customer_group as (
    select order_id, customer_id, order_date,
            row_number() over(partition by customer_id order by order_date) as rownum,
            date_sub(order_date, rownum) as group_date
        from customer_purchase
            where purchase_per_day = 1
),
customer_purchase_days as (
    select customer_id,
        min(order_date) as streak_start, 
        max(order_date) as streak_end,
        count(group_date) as continious_days
    from customer_group
        group by customer_id, group_date
            having count(group_date) >= 3
)
select c.customer_id, 
        c.customer_name,
        cpd.streak_start, 
        cpd.streak_end,
        cpd.continious_days
    from customer_purchase_days as cpd
        inner join sql_interview.customers as c
            on cpd.customer_id = c.customer_id

**Q059 ⭐⭐⭐⭐⭐⭐ (Customer Repeat Purchase Analysis)**

This is another classic **advanced window function** interview question.

**Business Scenario**

The CRM team wants to identify loyal customers.

A loyal customer is defined as someone who has placed **at least 3 Delivered orders within any rolling 30-day period**.

Unlike Q058 (consecutive days), the purchases **do not have to be on consecutive dates**.

**Business Rules**
* Consider only Delivered orders.
* If a customer places multiple Delivered orders on the **same day**, each order counts separately.
* A customer may satisfy the condition multiple times.
* Return **only the first qualifying 30-day window** for each customer.

**Return**
* Column:	Description
* customer_id:	Customer ID
* customer_name:	Customer Name
* window_start:	Date of the first order in the qualifying window
* window_end:	Date of the third (or later) order that completes the qualifying window
* orders_in_window:	Number of delivered orders in that qualifying 30-day window

**Difficulty**

⭐⭐⭐⭐⭐⭐⭐ (7.5/10)

This type of question is commonly asked in product analytics, customer retention, and e-commerce interviews because it requires you to think in terms of **rolling time windows** rather than simple grouping or ranking.

Good luck! This one will stretch your window-function skills in a different direction than Q058.

In [0]:
%sql

with customer_purchase as (
    select order_id, customer_id, order_date as window_start,
            max(order_date) over(partition by customer_id order by order_date
                                range between current row and interval 30 days following) as window_end,
            count(order_id) over(partition by customer_id order by order_date
                                range between current row and interval 30 days following) as orders_in_30_days
            
        from sql_interview.orders
            where order_status = 'Delivered'
),
ranked_windows as (
   select customer_id, window_start, window_end, orders_in_30_days as orders_in_window,
                   row_number() over(partition by customer_id order by orders_in_30_days desc, window_start asc) as rnk
        from customer_purchase cp
            where orders_in_30_days >= 3
        
)
select c.customer_id, c.customer_name, rw.window_start, rw.window_end, rw.orders_in_window, rnk
    from ranked_windows rw
    inner join sql_interview.customers c
        on rw.customer_id = c.customer_id
    where rnk = 1

       
        

In [0]:
%sql

with customer_purchase as (
    select order_id, customer_id, order_date as window_start,
            max(order_date) over(partition by customer_id order by order_date
                                range between current row and interval 30 days following) as window_end,
            count(order_id) over(partition by customer_id order by order_date
                                range between current row and interval 30 days following) as orders_in_30_days
            
        from sql_interview.orders
            where order_status = 'Delivered'
),
max_ordrer as (
   select distinct customer_id, window_start, window_end, orders_in_30_days as orders_in_window
    from customer_purchase cp
        where orders_in_30_days in (select max(orders_in_30_days) from customer_purchase where cp.customer_id = customer_id group by customer_id)
            and orders_in_30_days >= 3
)
select c.customer_id, c.customer_name, mo.window_start, mo.window_end, mo.orders_in_window
    from max_ordrer mo
    inner join sql_interview.customers c
        on mo.customer_id = c.customer_id


**Q060** ⭐⭐⭐⭐⭐⭐⭐ (**Amazon / Walmart / Databricks Level**)
**Business Scenario**

The Finance team has noticed that some customers repeatedly buy the **same product**.

They want to identify customers who **repurchased the same product within 30 days**.

**This is often used to measure**:

* Customer loyalty
* Product stickiness
* Subscription-like buying behavior

**Business Rules**: 
1. Consider only Delivered orders.
2. A repurchase means the same customer buys the same product again.
3. The second purchase must occur within 30 days of the previous purchase.
4. If a customer buys the same product 5 times, report every qualifying pair.
5. Ignore purchases made after more than 30 days.
6. Multiple products should be evaluated independently.


**Return**:
Column	Description
1. customer_id	Customer ID
2. customer_name	Customer Name
3. product_id	Product ID
4. product_name	Product Name
5. first_purchase_date	Earlier purchase date
6. second_purchase_date	Next purchase date
7. days_between	Difference in days
8. purchase_number	1st repurchase, 2nd repurchase, etc.

In [0]:
%sql
with customers_orders as (
    select o.customer_id, oi.product_id, o.order_date as first_purchase_date,
        lead(order_date) over(partition by customer_id, product_id order by order_date asc) as second_purchase_date,
        row_number() over(partition by o.customer_id, oi.product_id order by o.order_date asc) as purchase_number
        
        from sql_interview.orders o
            inner join sql_interview.order_items oi
                on o.order_id = oi.order_id
        where o.order_status = 'Delivered'
),
products_purchased as (
    select customer_id, product_id, first_purchase_date, second_purchase_date,
        date_diff(second_purchase_date, first_purchase_date) as days_between,
        purchase_number
    from customers_orders
        where second_purchase_date is not null
)
select pp.customer_id, 
        c.customer_name, 
        pp.product_id, 
        p.product_name, 
        first_purchase_date, 
        second_purchase_date,
        days_between, 
        purchase_number 
    from products_purchased pp
        inner join sql_interview.customers c
            on pp.customer_id = c.customer_id
        
        inner join sql_interview.products p
            on pp.product_id = p.product_id

    where days_between <= 30


In [0]:
%sql
select *
    from sql_interview.order_items
        

In [0]:
def first_non_repeating_char(s: str):
    _counts = {}

    for c in s:
        if c not in _counts:
            _counts[c] = 1
        else:
            _counts[c] +=1

    for c in s:
        if _counts[c] == 1:
            return c


if __name__ == "__main__":
    
    fc = first_non_repeating_char("aabbcc")
    print(fc)

In [0]:
str.lstrip?